# Baseline Age Model Training (LDAE, Combined Male + Female)

In [11]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DATA_DIR = Path("/content/data")
    CHECKPOINT_DIR = Path("/content/checkpoints/baseline_age_model")
else:
    NOTEBOOK_DIR = Path(".").resolve()
    PROJECT_ROOT = NOTEBOOK_DIR.parent
    DATA_DIR = PROJECT_ROOT / "data"
    CHECKPOINT_DIR = PROJECT_ROOT / "models" / "baseline_age_model"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

IMAGES_DIR = DATA_DIR / "utk_face"
TRAIN_AGE_MALE_CSV = DATA_DIR / "splits" / "train_age_male.csv"
TRAIN_AGE_FEMALE_CSV = DATA_DIR / "splits" / "train_age_female.csv"
VAL_AGE_MALE_CSV = DATA_DIR / "splits" / "val_age_male.csv"
VAL_AGE_FEMALE_CSV = DATA_DIR / "splits" / "val_age_female.csv"
TEST_CSV = DATA_DIR / "splits" / "test.csv"

BEST_MODEL = CHECKPOINT_DIR / "best_baseline_age_model.keras"

print(f"Running in Colab: {IN_COLAB}")
print(f"Data dir        : {DATA_DIR}")
print(f"Images dir      : {IMAGES_DIR}")
print(f"Checkpoint dir  : {CHECKPOINT_DIR}")


Running in Colab: True
Data dir        : /content/data
Images dir      : /content/data/utk_face
Checkpoint dir  : /content/checkpoints/baseline_age_model


In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

print(f"TensorFlow version : {tf.__version__}")
print(f"GPUs available     : {tf.config.list_physical_devices('GPU')}")

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-3
DROPOUT_RATE = 0.3
DENSE_UNITS = 256
NUM_AGE_BINS = 117
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)


TensorFlow version : 2.19.0
GPUs available     : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
def parse_age_distribution(dist_str: str) -> np.ndarray:
    values = np.fromstring(dist_str, sep=",", dtype=np.float32)
    if values.size != NUM_AGE_BINS:
        raise ValueError(
            f"Expected {NUM_AGE_BINS} LDAE bins, got {values.size}."
        )
    total = float(values.sum())
    if total <= 0:
        raise ValueError("Invalid LDAE distribution with non-positive sum.")
    return values / total


def load_age_dataframe(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["filepath"] = df["filename"].apply(lambda fname: str(IMAGES_DIR / fname))
    df["age_distribution_arr"] = df["age_distribution"].apply(parse_age_distribution)
    return df[["filepath", "age", "age_distribution_arr"]]


train_male_df = load_age_dataframe(TRAIN_AGE_MALE_CSV)
train_female_df = load_age_dataframe(TRAIN_AGE_FEMALE_CSV)
val_male_df = load_age_dataframe(VAL_AGE_MALE_CSV)
val_female_df = load_age_dataframe(VAL_AGE_FEMALE_CSV)
test_df = load_age_dataframe(TEST_CSV)

train_df = pd.concat([train_male_df, train_female_df], ignore_index=True)
val_df = pd.concat([val_male_df, val_female_df], ignore_index=True)

print(f"Train samples: {len(train_df)}")
print(f"Val samples  : {len(val_df)}")
print(f"Test samples : {len(test_df)}")


Train samples: 14104
Val samples  : 6045
Test samples : 3556


In [6]:
AUTOTUNE = tf.data.AUTOTUNE


def parse_image(filepath: tf.Tensor, dist: tf.Tensor):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = preprocess_input(image)
    return image, dist


def augment(image: tf.Tensor, dist: tf.Tensor):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.15)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    return image, dist


def make_dataset(df: pd.DataFrame, training: bool = False) -> tf.data.Dataset:
    filepaths = df["filepath"].to_numpy()
    dists = np.stack(df["age_distribution_arr"].to_numpy()).astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((filepaths, dists))

    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(parse_image, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

print("Datasets built successfully.")


Datasets built successfully.


In [7]:
class ExpectedAgeMAE(keras.metrics.Metric):
    """MAE between expected age from predicted vs true LDAE distributions."""

    def __init__(self, name: str = "expected_age_mae", **kwargs):
        super().__init__(name=name, **kwargs)
        self.total_abs_error = self.add_weight(name="total_abs_error", initializer="zeros")
        self.count = self.add_weight(name="count", initializer="zeros")
        self.age_bins = tf.cast(tf.range(NUM_AGE_BINS), tf.float32)

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        true_age = tf.reduce_sum(y_true * self.age_bins, axis=-1)
        pred_age = tf.reduce_sum(y_pred * self.age_bins, axis=-1)
        abs_err = tf.abs(true_age - pred_age)

        if sample_weight is not None:
            sample_weight = tf.cast(sample_weight, tf.float32)
            abs_err = abs_err * sample_weight
            batch_count = tf.reduce_sum(sample_weight)
        else:
            batch_count = tf.cast(tf.size(abs_err), tf.float32)

        self.total_abs_error.assign_add(tf.reduce_sum(abs_err))
        self.count.assign_add(batch_count)

    def result(self):
        return tf.math.divide_no_nan(self.total_abs_error, self.count)

    def reset_state(self):
        self.total_abs_error.assign(0.0)
        self.count.assign(0.0)


def build_model(
    image_size: tuple = IMAGE_SIZE,
    dense_units: int = DENSE_UNITS,
    dropout_rate: float = DROPOUT_RATE,
    learning_rate: float = LEARNING_RATE,
) -> keras.Model:
    backbone = ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=(*image_size, 3),
        name="resnet50",
    )
    backbone.trainable = False

    inputs = keras.Input(shape=(*image_size, 3), name="image_input")
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(dense_units, activation="relu", name="fc1")(x)
    x = layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = layers.Dense(NUM_AGE_BINS, activation="softmax", name="age_distribution_output")(x)

    model = keras.Model(inputs, outputs, name="baseline_age_model_estimator")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.KLDivergence(),
        metrics=[ExpectedAgeMAE()],
    )
    return model


if BEST_MODEL.exists():
    model = keras.models.load_model(
        str(BEST_MODEL),
        custom_objects={"ExpectedAgeMAE": ExpectedAgeMAE},
    )
    print(f"Resumed from checkpoint: {BEST_MODEL}")
else:
    model = build_model()
    print("No checkpoint found, starting from scratch.")

model.summary()


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
No checkpoint found, starting from scratch.


Model: "baseline_age_model_estimator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ age_distribution_output (Dense) │ (None, 117)            │        30,069 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,142,325 (92.10 MB)

 Trainable params: 554,613 (2.12 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [8]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL),
        monitor="val_expected_age_mae",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_expected_age_mae",
        mode="min",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

print(f"\nStage-1 training complete. Best model saved to: {BEST_MODEL}")


Epoch 1/30
441/441 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - expected_age_mae: 12.2227 - loss: 1.5601
Epoch 1: val_expected_age_mae improved from inf to 7.52332, saving model to /content/checkpoints/baseline_age_model/best_baseline_age_model.keras
441/441 ━━━━━━━━━━━━━━━━━━━━ 87s 166ms/step - expected_age_mae: 12.2173 - loss: 1.5596 - val_expected_age_mae: 7.5233 - val_loss: 1.1147 - learning_rate: 0.0010
Epoch 2/30
441/441 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - expected_age_mae: 7.8437 - loss: 1.1528
Epoch 2: val_expected_age_mae improved from 7.52332 to 7.12012, saving model to /content/checkpoints/baseline_age_model/best_baseline_age_model.keras
441/441 ━━━━━━━━━━━━━━━━━━━━ 64s 146ms/step - expected_age_mae: 7.8435 - loss: 1.1527 - val_expected_age_mae: 7.1201 - val_loss: 1.0642 - learning_rate: 0.0010
Epoch 3/30
441/441 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - expected_age_mae: 7.4065 - loss: 1.0884
Epoch 3: val_expected_age_mae improved from 7.12012 to 6.85303, saving model to /content/checkp

In [9]:
FINETUNE_LR = 1e-5
FINETUNE_EPOCHS = 20
FINETUNE_PATIENCE = 6

ft_model = keras.models.load_model(
    str(BEST_MODEL),
    custom_objects={"ExpectedAgeMAE": ExpectedAgeMAE},
)
print(f"Loaded checkpoint from: {BEST_MODEL}")

backbone = ft_model.get_layer("resnet50")
backbone.trainable = True

for layer in backbone.layers:
    if not layer.name.startswith("conv5"):
        layer.trainable = False

for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

ft_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FINETUNE_LR),
    loss=keras.losses.KLDivergence(),
    metrics=[ExpectedAgeMAE()],
)

ft_callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL),
        monitor="val_expected_age_mae",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_expected_age_mae",
        mode="min",
        patience=FINETUNE_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

ft_history = ft_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    callbacks=ft_callbacks,
)

print(f"\nFine-tuning complete. Best model saved to: {BEST_MODEL}")


Loaded checkpoint from: /content/checkpoints/baseline_age_model/best_baseline_age_model.keras
Epoch 1/20
441/441 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - expected_age_mae: 5.4662 - loss: 0.8057
Epoch 1: val_expected_age_mae improved from inf to 6.72136, saving model to /content/checkpoints/baseline_age_model/best_baseline_age_model.keras
441/441 ━━━━━━━━━━━━━━━━━━━━ 109s 210ms/step - expected_age_mae: 5.4662 - loss: 0.8057 - val_expected_age_mae: 6.7214 - val_loss: 1.0508 - learning_rate: 1.0000e-05
Epoch 2/20
441/441 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - expected_age_mae: 5.0624 - loss: 0.7599
Epoch 2: val_expected_age_mae improved from 6.72136 to 6.55685, saving model to /content/checkpoints/baseline_age_model/best_baseline_age_model.keras
441/441 ━━━━━━━━━━━━━━━━━━━━ 79s 179ms/step - expected_age_mae: 5.0625 - loss: 0.7599 - val_expected_age_mae: 6.5568 - val_loss: 1.0305 - learning_rate: 1.0000e-05
Epoch 3/20
441/441 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - expected_age_mae: 4.7900 - loss: 

In [10]:
best_model = keras.models.load_model(
    str(BEST_MODEL),
    custom_objects={"ExpectedAgeMAE": ExpectedAgeMAE},
)

eval_metrics = best_model.evaluate(test_ds, verbose=1)
metric_names = best_model.metrics_names

print("\nTest metrics:")
for name, value in zip(metric_names, eval_metrics):
    print(f"{name}: {value:.6f}")

age_bins = np.arange(NUM_AGE_BINS, dtype=np.float32)
pred_dist = best_model.predict(test_ds, verbose=1)
pred_age = (pred_dist * age_bins[None, :]).sum(axis=1)
true_age = test_df["age"].to_numpy(dtype=np.float32)
manual_mae = np.mean(np.abs(pred_age - true_age))
print(f"Manual decoded-age MAE: {manual_mae:.4f}")


112/112 ━━━━━━━━━━━━━━━━━━━━ 17s 118ms/step - expected_age_mae: 5.9761 - loss: 1.1426

Test metrics:
loss: 1.150753
compile_metrics: 5.938206
112/112 ━━━━━━━━━━━━━━━━━━━━ 19s 135ms/step
Manual decoded-age MAE: 5.9479
